<a href="https://colab.research.google.com/github/NeerajMann19/PAX-hybrid-chatbot/blob/main/PAX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# FULL HYBRID CHATBOT — now with Emotion detection + Math Solver + Decision support
# Install only if needed (uncomment if running fresh):
# !pip install -q datasets scikit-learn joblib gradio transformers torch sympy matplotlib

import os
import random
import re
import joblib
import pandas as pd
import numpy as np
import torch
import gradio as gr
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sympy import symbols, Eq, solve, simplify
import sympy as sp
import io
import base64
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings('ignore')

LABEL_MAP = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise", 6: "neutral"}
EMOJI_MAP = {"joy": "😄", "sadness": "😢", "anger": "😡", "fear": "😨", "surprise": "😲", "love": "❤", "neutral": "💬"}
EMOTIONAL_REPLIES = {
    "joy": ["That’s wonderful! Keep smiling 😄", "I'm so glad to hear that! 🌟"],
    "sadness": ["I’m sorry you’re feeling down 💙", "It's okay to be sad. Sending hugs 🤗"],
    "anger": ["I understand you’re upset 😔", "Take a deep breath. It will be okay 🌱"],
    "fear": ["It’s okay to feel scared. You’re not alone 🌸", "I'm here for you."],
    "surprise": ["Wow! That’s unexpected 👀", "Really?! That’s amazing! 🎉"],
    "love": ["That’s so sweet ❤", "Love makes the world go round 💕"],
    "neutral": ["Gotcha. Tell me more.", "Okay — what's next?"]
}
DECISION_PERSONA = """You are a chill, supportive chatbot 😎. For decision questions, always give a clear recommendation (e.g., 'I recommend [option] because...') based on the user's context like mood, weather, event type, or details provided. Explain why briefly, keep it positive and casual, and end with an emoji. Be helpful and specific—avoid vague answers."""
MAX_CONTEXT_TOKENS = 900
GEN_KWARGS = dict(do_sample=True, top_k=50, top_p=0.9, temperature=0.7, max_new_tokens=120)
VEC_PATH = "tfidf_vectorizer.joblib"
CLF_PATH = "emotion_classifier.joblib"
BANNED_WORDS = {"badword1", "badword2"}

def safe_text(s: str) -> str:
    s = s or ""
    def _sub(m): return "*" * len(m.group(0))
    pattern = re.compile(r"|".join(re.escape(w) for w in BANNED_WORDS), flags=re.IGNORECASE) if BANNED_WORDS else None
    if pattern: s = pattern.sub(_sub, s)
    return re.sub(r"[\x00-\x08\x0b-\x1f]+", " ", s).strip()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Train / load emotion classifier ---
if not (os.path.exists(VEC_PATH) and os.path.exists(CLF_PATH)):
    print("Training emotion SVM classifier (may take a few minutes)...")
    dataset = load_dataset("emotion", trust_remote_code=True)
    df = pd.concat([pd.DataFrame(dataset[s]) for s in ['train','test','validation']], ignore_index=True)
    neutral_texts = ["The weather is calm today.", "I'm just browsing.", "Let me check my schedule.", "The meeting is at 3 PM.", "Okay, I will do that.", "Can you pass the salt?"]
    neutral_df = pd.DataFrame({'text': neutral_texts, 'label': [6]*len(neutral_texts)})
    df = pd.concat([df, neutral_df], ignore_index=True)
    df['label_name'] = df['label'].map(LABEL_MAP)
    df_balanced = pd.concat([
        df[df['label_name']=='joy'].sample(n=min(3000,len(df[df['label_name']=='joy'])),random_state=42),
        df[df['label_name']=='sadness'].sample(n=min(3000,len(df[df['label_name']=='sadness'])),random_state=42),
        df[df['label_name'].isin(['love','anger','fear','surprise','neutral'])]
    ]).sample(frac=1, random_state=42)
    X = df_balanced['text']; y = df_balanced['label']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    vec = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1,2))
    X_train_vec = vec.fit_transform(X_train)
    svm = LinearSVC(class_weight='balanced', dual=False, max_iter=2000)
    clf = CalibratedClassifierCV(svm)
    clf.fit(X_train_vec, y_train)
    joblib.dump(vec, VEC_PATH)
    joblib.dump(clf, CLF_PATH)
    print("Saved emotion models.")
else:
    print("Found existing emotion models; skipping training.")

loaded_vec = joblib.load(VEC_PATH)
loaded_clf = joblib.load(CLF_PATH)
print("Emotion models loaded.")

print("Loading DialoGPT (may download weights on first run)...")
generative_tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
generative_model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium").to(device)
if generative_tokenizer.pad_token is None:
    generative_tokenizer.pad_token = generative_tokenizer.eos_token
generative_model.eval()
print("DialoGPT ready.")

# --- Utility functions ---
def is_decision_question(text: str) -> bool:
    t=text.lower()
    return any(k in t for k in ["should i","should we","decide for me","choose between","which should i","help me choose"]) or (" or " in t and "?" in t and len(t.split())<30)

def is_math_query(text: str) -> bool:
    t = text.lower().strip()
    math_keywords = ["calculate", "solve", "what is", "find", "equation", "parabola", "graph", "roots", "vertex", "integral", "derivative"]
    if any(kw in t for kw in math_keywords):
        return True
    math_pattern = r'[\d+\-*/().=xXyYzZ^ ]{3,}'
    if re.search(math_pattern, t):
        return True
    return False

def solve_math_problem(text: str) -> tuple[str, str | None]:
    t = text.strip()
    try:
        if re.match(r'^[\d\s+\-*/().=]+$', t.replace(' ', '')):
            allowed_names = {"__builtins__": {}}
            result = eval(t, allowed_names)
            return f"Calculation: {t} = {result}", None
        x = symbols('x')
        y = symbols('y')
        if "solve" in t.lower() or "roots" in t.lower():
            expr_str = re.sub(r'(solve|roots of|find roots of)\s+', '', t.lower(), flags=re.IGNORECASE)
            expr_str = expr_str.replace('=', '-').replace('for x', '')
            try:
                expr = sp.sympify(expr_str)
                roots = solve(expr, x)
                return f"Solving {t}:\nRoots: {roots}", None
            except:
                pass
        if "parabola" in t.lower() or "graph" in t.lower() or "y =" in t:
            expr_str = re.sub(r'(graph|parabola)\s+|y\s*=\s*', '', t.lower(), flags=re.IGNORECASE)
            try:
                expr = sp.sympify(expr_str)
                return f"Parabola: y = {expr}", None
            except:
                pass
        expr = sp.sympify(t)
        simplified = simplify(expr)
        return f"Simplified: {t} → {simplified}", None
    except Exception as e:
        return f"Sorry, couldn't solve that. Try again! (Error: {str(e)[:50]})", None

def generate_reply_from_model(prefix_text: str, user_text: str, history_ids: torch.Tensor):
    combined_text = (prefix_text + " " + user_text + generative_tokenizer.eos_token).strip()
    new_input_ids = generative_tokenizer.encode(combined_text, return_tensors='pt').to(device)
    if history_ids is None:
        bot_input_ids = new_input_ids
    else:
        history_on_device = history_ids.to(device)
        bot_input_ids = torch.cat([history_on_device, new_input_ids], dim=-1)
    bot_input_ids = trim_history_ids(bot_input_ids, MAX_CONTEXT_TOKENS).to(device)
    with torch.no_grad():
        generated_ids = generative_model.generate(bot_input_ids, **GEN_KWARGS, pad_token_id=generative_tokenizer.eos_token_id)
    reply_ids = generated_ids[:, bot_input_ids.shape[-1]:]
    reply = generative_tokenizer.decode(reply_ids[0], skip_special_tokens=True).strip()
    new_history = trim_history_ids(torch.cat([bot_input_ids, reply_ids], dim=-1), MAX_CONTEXT_TOKENS)
    return reply, new_history.cpu()

def get_decision_fallback(text: str) -> str:
    t = text.lower()
    if "wear" in t and ("black" in t or "white" in t) and "sunny" in t:
        return "I recommend the white shirt for a sunny day—it's lighter and cooler!"
    return "Based on your query, I recommend the first option."

def trim_history_ids(history_ids: torch.Tensor, max_tokens: int):
    if history_ids is None: return None
    total_len = history_ids.shape[-1]
    return history_ids if total_len <= max_tokens else history_ids[:, -max_tokens:]

# --- NEW EMOTION HANDLING ADDITION ---
def classify_emotion(text: str) -> str:
    """Predict emotion label with keyword correction rules for realism."""
    try:
        v = loaded_vec.transform([text])
        pred = loaded_clf.predict(v)[0]
        label = LABEL_MAP.get(int(pred), "neutral")
    except Exception:
        label = "neutral"

    t = text.lower()

    # --- manual keyword corrections ---
    negative_words = ["not", "sad", "upset", "bad", "tired", "sick", "hurt", "depressed", "lonely", "cry", "low", "down"]
    if any(w in t for w in negative_words):
        if label in {"joy", "love", "surprise", "neutral"}:
            label = "sadness"

    if any(w in t for w in ["angry", "mad", "furious", "annoyed"]):
        label = "anger"
    if any(w in t for w in ["afraid", "scared", "terrified", "nervous", "worried", "anxious"]):
        label = "fear"

    return label



def smart_chatbot(text: str, history_ids):
    text = (text or "").strip()
    if not text:
        return "neutral", "Say something so I can help 😄", history_ids
    if text.lower() in {"hi","hello","hey","yo"}:
        return "neutral", "Hello there! How are you feeling today?", history_ids

    # math queries
    if is_math_query(text):
        explanation, _ = solve_math_problem(text)
        return "neutral", explanation, history_ids

    # decision questions
    if is_decision_question(text):
        use_history = history_ids if any(kw in text.lower() for kw in ["what should i", "why"]) else None
        reply_text, new_history_ids = generate_reply_from_model(DECISION_PERSONA, text, use_history)
        if (len(reply_text) < 20 or any(phrase in reply_text.lower() for phrase in ["rules of this sub"])):
            reply_text = get_decision_fallback(text)
        return "neutral", reply_text, new_history_ids

    # --- emotion path ---
    emotion_label = classify_emotion(text)
    empathetic = choose_emotional_reply(emotion_label, text)

    follow_up = ""
    if emotion_label in {"sadness", "fear", "anger"}:
        follow_up = " Would you like to talk about what's bothering you?"
    elif emotion_label in {"joy","surprise","love"}:
        follow_up = " Tell me more — that sounds great!"
    reply = f"{empathetic}{follow_up}"

    return emotion_label, reply, history_ids


# --- Gradio UI ---
demo = gr.Blocks()
with demo:
    gr.Markdown("# 💬 Hybrid Chatbot — with Emotion, Math, and Decisions")
    chatbot = gr.Chatbot()
    txt = gr.Textbox(placeholder="Type here...")
    btn = gr.Button("Send")
    btn.click(chatbot_for_gradio,
              inputs=[txt, chatbot, gr.State(), gr.State()],
              outputs=[chatbot, gr.State(), gr.State(), gr.Textbox(), txt])
    txt.submit(chatbot_for_gradio,
               inputs=[txt, chatbot, gr.State(), gr.State()],
               outputs=[chatbot, gr.State(), gr.State(), gr.Textbox(), txt])

demo.launch()


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Training emotion SVM classifier (may take a few minutes)...


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

DialoGPT ready.


NameError: name 'chatbot_for_gradio' is not defined